In [ ]:
import pandas as pd

financial_df = pd.read_csv("./csv/customer_financial_health_with_ai.csv")
customers_df = pd.read_csv("./csv/customers.csv")
purchases_df = pd.read_csv("./csv/home_purchases.csv")
loans_df = pd.read_csv("./csv/loans_raw.csv")
estate_detail_df = pd.read_csv("./csv/real_estate_detail_raw.csv")
estate_df = pd.read_csv("./csv/real_estate_raw.csv")


In [ ]:
print("financial")
display(financial_df.head())

print("customers")
display(customers_df.head())

print("purchases")
display(purchases_df.head())

print("loans")
display(loans_df.head())

print("estate_detail")
display(estate_detail_df.head())

print("estate")
display(estate_df.head())

In [ ]:
dfs = {
    "customer_financial_health_with_ai": financial_df,
    "customers": customers_df,
    "home_purchases": purchases_df,
    "loans_raw": loans_df,
    "real_estate_detail_raw": estate_detail_df,
    "real_estate_raw": estate_df
}

for name, df in dfs.items():
    print("=" * 100)
    print(f"테이블명: {name}")
    
    print("\n[1. 데이터 크기]")
    print(df.shape)
    
    print("\n[2. 컬럼명 / 데이터 타입]")
    print(df.dtypes)
    
    print("\n[3. 데이터 샘플]")
    display(df.head(5))
    
    print("\n[4. 컬럼별 고유값 개수]")
    print(df.nunique())
    
    print("\n")

 for name, df in dfs.items():
    print(f"\n===== {name} =====")
    print(df.columns.tolist())

In [ ]:
loans_df["loan_purpose"].unique()

In [ ]:
# customers
customer_cols = [
    "primary_annual_income",
    "household_annual_income",
    "avg_monthly_primary_income",
    "total_financial_debt",
    "mortgage_balance"
]

print("=== customers ===")
print(customers_df[customer_cols].isnull().sum())


# home_purchases
purchase_cols = [
    "primary_annual_income",
    "household_annual_income",
    "total_loan_principal",
    "mortgage_principal",
    "other_loan_principal",
    "bought_with_loan"
]

print("\n=== home_purchases ===")
print(purchases_df[purchase_cols].isnull().sum())


# loans_raw
loan_cols = [
    "original_principal",
    "principal",
    "outstanding_balance",
    "housing_purchase_linked"
]

print("\n=== loans_raw ===")
print(loans_df[loan_cols].isnull().sum())

In [ ]:
loan_null_check = purchases_df[
    [
        "total_loan_principal",
        "mortgage_principal",
        "other_loan_principal"
    ]
].isnull()

loan_null_check.value_counts()

In [ ]:
# 대출금 NULL인 사람들의 대출 이용 여부
purchases_df.loc[
    purchases_df["total_loan_principal"].isnull(),
    "bought_with_loan"
].value_counts(dropna=False)

In [ ]:
# 대출 사용 여부별 대출금 상태
purchases_df.groupby("bought_with_loan")[
    ["total_loan_principal",
     "mortgage_principal",
     "other_loan_principal"]
].agg(["count", "min", "max"])

In [ ]:
# home_purchase만 null값 존재. 따라서 0처리
loan_amount_cols = [
    "total_loan_principal",
    "mortgage_principal",
    "other_loan_principal"
]

purchases_df[loan_amount_cols] = (
    purchases_df[loan_amount_cols].fillna(0)
)
purchases_df[loan_amount_cols].isnull().sum()

In [ ]:
# 1. 대출 0 + 개인 연소득 0
zero_income_loan = purchases_df[
    (purchases_df["total_loan_principal"] == 0) &
    (purchases_df["primary_annual_income"] == 0)
].copy()

print("대출 0 + 개인 연소득 0:", len(zero_income_loan))

In [ ]:
zero_income_loan[
    [
        "customer_id",
        "age",
        "employment_type",
        "primary_annual_income",
        "household_annual_income",
        "financial_assets_estimated",
        "purchase_price_est",
        "total_loan_principal",
        "own_funds_est"
    ]
].head(20)

home purchase 기준으로는 대출이 0인데 소득이 0인경우 없음.

In [ ]:
no_loan_ids = purchases_df.loc[
    purchases_df["total_loan_principal"] == 0,
    "customer_id"
]

check_customers = customers_df[
    customers_df["customer_id"].isin(no_loan_ids)
]

income_cols = [
    "primary_monthly_income",
    "primary_annual_income",
    "household_annual_income",
    "disposable_annual_income",
    "household_monthly_income",
    "avg_monthly_primary_income"
]

for col in income_cols:
    print(f"{col} = 0 : {(check_customers[col] == 0).sum()}명")

In [ ]:
no_loan_purchases = purchases_df[
    purchases_df["total_loan_principal"] == 0
][["customer_id", "real_asset_id"]]

loan_check = no_loan_purchases.merge(
    loans_df[
        [
            "customer_id",
            "reference_real_asset_id",
            "original_principal",
            "loan_purpose",
            "housing_purchase_linked"
        ]
    ],
    left_on=["customer_id", "real_asset_id"],
    right_on=["customer_id", "reference_real_asset_id"],
    how="left"
)

loan_check.head()

In [ ]:
loan_check["loan_check_result"] = "기타"

# loans_raw에 매칭되는 대출 자체가 없음
loan_check.loc[
    loan_check["original_principal"].isna(),
    "loan_check_result"
] = "대출기록 없음"

# 대출은 있지만 주택구매와 연결되지 않음
loan_check.loc[
    loan_check["original_principal"].notna() &
    (loan_check["housing_purchase_linked"] == 0),
    "loan_check_result"
] = "대출 있음 / 주택구매 비연결"

# 주택구매 연결 대출 존재
loan_check.loc[
    loan_check["original_principal"].notna() &
    (loan_check["housing_purchase_linked"] == 1),
    "loan_check_result"
] = "주택구매 연결대출 있음"

loan_check["loan_check_result"].value_counts()

## home purchases 에서 대출없이 구매 11920건
## 9592건은 두 테이블에서 대출기록이 확인안됨
## 2328건은 대출기록은 있지만 주택구매와 무관

In [ ]:
# 20대 무대출 구매건
twenties_no_loan = purchases_df[
    (purchases_df["age_at_purchase"].between(20, 29)) &
    (purchases_df["bought_with_loan"] == 0)
].copy()

print("20대 무대출 주택구매 건수:", len(young_no_loan))
print("고객 수:", young_no_loan["customer_id"].nunique())

In [ ]:
income_cols = [
    "primary_monthly_income",
    "primary_annual_income",
    "household_annual_income",
    "disposable_annual_income",
    "household_monthly_income",
    "avg_monthly_primary_income"
]

for col in income_cols:
    print(f"{col} = 0 :", (y_no_loan[col] == 0).sum())

In [ ]:
#import pandas as pd
#pd.options.display.float_format = '{:,.0f}'.format

In [ ]:
twenties_no_loan[
    [
        "customer_id",
        "age_at_purchase",
        "employment_type",
        "primary_annual_income",
        "household_annual_income",
        "financial_assets_estimated",
        "purchase_price_est",
        "own_funds_est",
        "own_funds_ratio"
    ]
].sort_values(
    ["primary_annual_income", "financial_assets_estimated"]
).head(30)

In [ ]:
cols = [
    "age_at_purchase",
    "primary_annual_income",
    "household_annual_income",
    "financial_assets_estimated",
    "purchase_price_est",
    "own_funds_est",
    "own_funds_ratio"
]

twenties_no_loan[cols].describe().T

In [ ]:
twenties_no_loan["price_income_ratio"] = (
    twenties_no_loan["purchase_price_est"] /
    twenties_no_loan["primary_annual_income"]
)

twenties_no_loan["price_asset_ratio"] = (
    twenties_no_loan["purchase_price_est"] /
    twenties_no_loan["financial_assets_estimated"]
)

twenties_no_loan[
    [
        "price_income_ratio",
        "price_asset_ratio",
        "own_funds_ratio"
    ]
].describe()

In [ ]:
asset_enough = (
    young_no_loan["financial_assets_estimated"]
    >= young_no_loan["purchase_price_est"]
)

print("금융자산 >= 집값 :", asset_enough.sum())
print("금융자산 < 집값 :", (~asset_enough).sum())

print(
    "금융자산으로 집값 충당 가능 비율:",
    round(asset_enough.mean() * 100, 2), "%"
)

소득으로는 집값이 충당되지 않음.